In [138]:
import pandas as pd
import numpy as np

test = pd.read_csv("test.csv")
test.head()
test.shape

(1459, 80)

In [18]:
train= pd.read_csv("train.csv")
train.head()
train.shape

(1460, 81)

In [6]:
%pip install ydata-profiling

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [7]:
from ydata_profiling import ProfileReport
data = pd.concat([train, test], ignore_index=True)
profile = ProfileReport(data, title="EDA Report", explorative=True)
profile.to_file("EDA_Report1.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 81/81 [00:02<00:00, 38.10it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [117]:
## concat both and fill NaN 

In [61]:
test["SalePrice"] = -1
data = pd.concat([train, test], ignore_index=True)

In [62]:
(data.isnull().mean() * 100).sort_values(ascending=False)

PoolQC          99.657417
Alley           93.216855
MasVnrType      60.500171
FireplaceQu     48.646797
LotFrontage     16.649538
                  ...    
CentralAir       0.000000
1stFlrSF         0.000000
2ndFlrSF         0.000000
LowQualFinSF     0.000000
SalePrice        0.000000
Length: 79, dtype: float64

In [63]:
missing_percent = (data.isnull().mean() * 100)
cols_above_5 = missing_percent[missing_percent > 5]
print(cols_above_5.sort_values(ascending=False))

PoolQC          99.657417
Alley           93.216855
MasVnrType      60.500171
FireplaceQu     48.646797
LotFrontage     16.649538
GarageYrBlt      5.447071
GarageFinish     5.447071
GarageQual       5.447071
GarageCond       5.447071
GarageType       5.378554
dtype: float64


In [64]:
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

PoolQC          2909
Alley           2721
MasVnrType      1766
FireplaceQu     1420
LotFrontage      486
GarageYrBlt      159
GarageCond       159
GarageFinish     159
GarageQual       159
GarageType       157
BsmtCond          82
BsmtExposure      82
BsmtQual          81
BsmtFinType2      80
BsmtFinType1      79
MasVnrArea        23
MSZoning           4
BsmtFullBath       2
BsmtHalfBath       2
Utilities          2
Functional         2
GarageCars         1
GarageArea         1
TotalBsmtSF        1
KitchenQual        1
Electrical         1
BsmtUnfSF          1
BsmtFinSF2         1
BsmtFinSF1         1
Exterior2nd        1
Exterior1st        1
SaleType           1
dtype: int64


In [66]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [67]:
none_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None"))
])

zero_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0))
])

mean_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

median_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

mode_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

In [110]:
mode_cols = ["SaleType", "Exterior1st", "Exterior2nd", "Electrical", "KitchenQual", "Functional", "Utilities", "MSZoning"]

none_cols = ["PoolQC", "Alley", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "FireplaceQu", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]

In [109]:
preprocessor1 = ColumnTransformer([
    ("none", none_pipeline, none_cols),
    ("mode", mode_pipeline, mode_cols)
], remainder="passthrough")
## havent used it but learnt...

In [76]:
data[none_cols] = data[none_cols].fillna("None")
data[mode_cols] = data[mode_cols].fillna(data[mode_cols].mode().iloc[0])

In [111]:
data.loc[data["TotalBsmtSF"] == 0,
         ["BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [112]:
data.loc[data["BsmtQual"] == "None" ,
         ["BsmtFinSF1","TotalBsmtSF", "BsmtFinSF2", "BsmtUnfSF",
          "BsmtFullBath", "BsmtHalfBath"]] = 0

In [113]:
data.loc[data["GarageType"] == "None",
         ["GarageArea","GarageCars"]] = 0

In [101]:
((data["MasVnrType"].isnull()) & (data["MasVnrArea"].isnull())).sum()

np.int64(0)

In [86]:
mask = data["MasVnrArea"].isnull() | (data["MasVnrArea"] == 0)
data.loc[mask, "MasVnrArea"] = 0
data.loc[mask, "MasVnrType"] = "None"

In [92]:
data.loc[
    data["MasVnrType"].isnull() & (data["MasVnrArea"] > 0),
    "MasVnrType"
] = data["MasVnrType"].mode()[0]

In [114]:
((data["GarageType"] == "None") & (data["GarageArea"].isnull())).sum()

np.int64(0)

In [98]:
data[data["GarageCars"].isnull()][["GarageType", "GarageArea", "GarageCars"]]

,GarageType,GarageArea,GarageCars
2576,Detchd,NaN,NaN


In [102]:
data["GarageCars"] = data["GarageCars"].fillna(data["GarageCars"].median())
data["GarageArea"] = data["GarageArea"].fillna(data["GarageArea"].median())

In [103]:
data["LotFrontage"] = data["LotFrontage"].fillna(data["LotFrontage"].median())

In [105]:
data.loc[data["GarageType"] == "None", "GarageYrBlt"] = 0

In [107]:
data["GarageYrBlt"] = data["GarageYrBlt"].fillna(data["GarageYrBlt"].median())

In [115]:
missing = data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)
## missing evaluated

Series([], dtype: int64)


In [118]:
## Handling categorial features

In [119]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [120]:
cat_cols = data.select_dtypes(include="object").columns
print(cat_cols)

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'SaleType', 'SaleCondition'],
      dtype='object')


In [121]:
ordinal_cols = ["LotShape", "Utilities", "LandSlope", "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "HeatingQC", "KitchenQual", "Functional", "FireplaceQu", "GarageFinish", "GarageQual", "GarageCond", "PavedDrive", "PoolQC"]

nominal_cols = ["MSZoning", "Street", "Alley", "LandContour", "LotConfig", "Neighborhood", "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", "Heating", "CentralAir", "Electrical", "GarageType", "SaleType", "SaleCondition"]

In [126]:
ordinal_encoder = OrdinalEncoder()
ohe_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
data[ordinal_cols] = ordinal_encoder.fit_transform(data[ordinal_cols])

encoded = ohe_encoder.fit_transform(data[nominal_cols])
encoded = pd.DataFrame(
    encoded,
    columns=ohe_encoder.get_feature_names_out(nominal_cols),
    index=data.index
)

data = data.drop(columns=nominal_cols)
data = pd.concat([data, encoded], axis=1)

In [127]:
data.shape

(2919, 221)

In [135]:
df1 = data[data["SalePrice"] != -1].copy()
df2 = data[data["SalePrice"] == -1].drop(columns="SalePrice").copy()

X = df1.drop(columns="SalePrice")
y = df1["SalePrice"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)
## Not my idea but to calculate accuracy this is best

In [129]:
X_train.shape

(1168, 220)

In [ ]:
## numerical vals scaling

In [132]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

In [133]:
print(num_cols)

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'LotShape', 'Utilities',
       'LandSlope', 'OverallQual', 'OverallCond', 'YearBuilt',
       ...
       'SaleType_ConLw', 'SaleType_New', 'SaleType_Oth', 'SaleType_WD',
       'SaleCondition_Abnorml', 'SaleCondition_AdjLand',
       'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal',
       'SaleCondition_Partial'],
      dtype='object', length=220)


In [136]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_val[num_cols])

df2[num_cols] = scaler.transform(df2[num_cols])

In [137]:
## linear regression

In [139]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_val, y_pred)))
print("MAE :", mean_absolute_error(y_val, y_pred))
print("R2  :", r2_score(y_val, y_pred))

RMSE: 140847213.46917716
MAE : 135424728.46969056
R2  : -2586321.5378138064


In [140]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(lr,X,y,cv=5,scoring="neg_root_mean_squared_error")

print("CV RMSE:", -cv_scores)
print("Mean CV RMSE:", -cv_scores.mean())

CV RMSE: [28657.36378796 33971.42342358 37299.81567668 26464.61796526
 49746.94990737]
Mean CV RMSE: 35228.0341521692


In [143]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
import numpy as np

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R2  :", r2_score(y_test, y_pred))

cv = cross_val_score(ridge, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("Mean CV RMSE:", -cv.mean())

RMSE: 125068053.8428279
MAE : 119269518.90724143
R2  : -2039288.8123910122
Mean CV RMSE: 32606.382850097947


In [145]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0, max_iter=100000)
lasso.fit(X_train, y_train)

y_pred = lasso.predict(X_val)

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("R2  :", r2_score(y_test, y_pred))

cv = cross_val_score(lasso, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("Mean CV RMSE:", -cv.mean())

RMSE: 131078597.24581349
MAE : 125554641.69028164
R2  : -2240007.859053603
Mean CV RMSE: 34484.2745954071


In [159]:
## tuning
ridge = Ridge(alpha=10)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_val)
cv = cross_val_score(ridge, X, y, cv=5, scoring="neg_root_mean_squared_error")
print("Mean CV RMSE:", -cv.mean())

Mean CV RMSE: 31945.254575632145
